Instalação de pacotes, seguido de importação de bibliotecas e funções necessárias.

In [ ]:
# Carregar as questões a serem usadas, cuja lógia está em outro arquivo.
# Realizar download do arquivo direto do GitHub.
!wget https://raw.githubusercontent.com/eduoududu/juridico/refs/heads/main/questions.ipynb -O questions.ipynb

# Executar o notebook de base na íntegra.
%run questions.ipynb

--2026-04-02 12:52:48--  https://raw.githubusercontent.com/eduoududu/juridico/refs/heads/main/questions.ipynb
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6565 (6.4K) [text/plain]
Saving to: ‘questions.ipynb’

questions.ipynb     100%[===================>]   6.41K  --.-KB/s    in 0s      

2026-04-02 12:52:48 (56.0 MB/s) - ‘questions.ipynb’ saved [6565/6565]



In [ ]:
df_questions_and_guidelines.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   question     10 non-null     int64 
 1   question_id  10 non-null     object
 2   category     10 non-null     object
 3   statement    10 non-null     object
 4   turns        10 non-null     object
 5   system       10 non-null     object
 6   choices      10 non-null     object
dtypes: int64(1), object(6)
memory usage: 692.0+ bytes


In [1]:
# Instalar ou atualizar bibliotecas necessárias: datasets do Hugging Face, transformers, torch e google.
#!pip install -q transformers accelerate bitsandbytes
!pip install -q -U google-genai

# Importar biblioteca pandas, função genai da biblioteca google e função userdata da biblioteca google.colab.
import pandas as pd
#import torch
from google import genai
from google.colab import userdata
#from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 760.6/760.6 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.7/240.7 kB 10.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.47.0, but you have google-auth 2.49.1 which is incompatible.


Google Gemini em nuvém, configuração, definição do modelo e execução da consulta.

In [ ]:
# Recuperar a chave da API de forma segura, armazenada no Secrets do Google Colab.
# O uso do Secrets é uma alternativa para que chave de API não fique exposta no código.
# Tal chave previamente criada no Google AI Studio.
# Observação: o nome da chave definido no Google AI Studio precisa ser exatamente o mesmo cadastrado no Secrets.
#             inclusive com diferenciação de letra maiúscula e minúscula.
google_api_key = userdata.get('google_api_key')

# Inicializar o cliente da API.
client_ai = genai.Client(api_key=google_api_key)

# O modelo escolhi do para rodar é o Gemini da Google em nuvém ena sua versão mais recente disponível, limitado gratuitamente à 20 requisições.
model_id = 'gemini-flash-latest'

# Criar uma lista vazia, para guardar as respostas, por questão de performance.
gemini_responses = []

# Repetição que percorre todo Dataframe.
for index, row in df_questions_and_guidelines.iterrows():
    # preenchimento dos parâmetros da pergunta, com base na linha corrente.
    questao = row['question']
    papel = row['system']
    categoria = row['category']
    contexto = row['statement']
    instrucao = row['turns']

    # Preparação do prompt em Markdown.
    prompt_usuario = f"""
    ### PAPEL:
    {papel}

    ### ÁREA:
    {categoria}

    ### CONTEXTO:
    {contexto}

    ### INSTRUÇÃO:
    {instrucao}
    """

    # Chamada simples para a API da Google em nuvem.
    response = client_ai.models.generate_content(
        model=model_id,
        contents=prompt_usuario,
        config={
            "temperature": 0.1,  # Conservador.
            "max_output_tokens": 1024
        }
    )

    # Armazenar o resultado em uma lista.
    gemini_responses.append({"question": questao, "response": response})
    print(f"Questão {questao} processada com sucesso.")

# Fechar conexão com a IA
client_ai.close()

# Para melhor visualização, converter as respostas em um Dataframe.
df_gemini_response = pd.DataFrame(gemini_responses)

ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

Realização de perguntas, uma por uma, por API. Em construção. Não execute.

In [ ]:
# Exibir as 5 primeiras linhas de respostas.
df_gemini_response.head()

In [ ]:
# Objetos mais relevantes até aqui.

# DataFrame com perguntas e respostas da linha guia.
df_question_and_guidelines.info()

# Dataframe com as respostas do gemini, com o campo question para relacionar com o dataframe das perguntas.
df_gemini_response.info()